<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook builds the Phase 1 dataset-summary table comparing Illumina, PacBio, and Oxford Nanopore cohorts. It pulls short-read and long-read QC metrics, filters withdrawals, joins Cohort Builder person and EHR counts, and plots coverage and mean read-length distributions.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas as pd
import numpy as np
import pysam
import os
import ast

## Participants who withdrew between CDR v7 and v8 

### The following participants were found to have withdrawn from the AoU Research Program between the releases of CDR v7 and v8. We list them here and filter them out of summaries and analyses.

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Controlled Tier withdrawal list (research IDs).
!gsutil cp {AOU_LR_DATA}/withdrawn_participants.txt withdrawn_participants.txt
withdrawn_participants = [int(line.strip()) for line in open("withdrawn_participants.txt") if line.strip()]

## Short read metrics

In [ ]:
# Dataset Builder queries: demographics and survey answers for every participant with short-read WGS.
import pandas
import os

# This query represents dataset "Short reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_37633348_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) )"""

dataset_37633348_person_df = pandas.read_gbq(
    dataset_37633348_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_37633348_person_df.head(5)

# This query represents dataset "Short reads" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_37633348_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) )
        )"""

dataset_37633348_survey_df = pandas.read_gbq(
    dataset_37633348_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_37633348_survey_df.head(5)
import pandas
import os

# This query represents dataset "Short reads" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_37633348_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_37633348_zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_37633348_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_37633348_zip_code_socioeconomic_df.head(5)

In [ ]:
# Per-sample QC metrics for the short-read callset (coverage, sex at birth); requester-pays bucket.
if not os.path.exists("genomic_metrics.tsv"):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv .
    
sr_metrics = pd.read_table('genomic_metrics.tsv')

In [ ]:
# Keep metrics only for participants returned by the query above.
sr_metrics = sr_metrics[sr_metrics['research_id'].isin(dataset_37633348_person_df['person_id'])]

In [ ]:
sr_metrics

In [ ]:
# Short-read column of the summary table: participant count, mean +/- SD coverage, and sex at birth.
sr_num_samples = f'{len(dataset_37633348_person_df):,}'
sr_num_samples

In [ ]:
sr_cov = f'{sr_metrics.mean_coverage.mean():.1f} +/- {sr_metrics.mean_coverage.std():.1f}'
sr_cov

In [ ]:
sr_female = sr_metrics.query('sex_at_birth == "F"').count()['research_id']
sr_male = sr_metrics.query('sex_at_birth == "M"').count()['research_id']
sr_unknown = len(dataset_37633348_person_df) - sr_female - sr_male

sr_female = f'{sr_female:,}'
sr_male = f'{sr_male:,}'
sr_unknown = f'{sr_unknown:,}'

(sr_female, sr_male, sr_unknown)

In [ ]:
# Short-read length: the length of the 100th read of one example CRAM (Illumina reads have a fixed length).
!gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/v7/wgs/cram/manifest.csv | head -n 2

In [ ]:
sr_read_lengths = !gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/pooled/wgs/cram/v6_base/wgs_example_sample.cram | samtools view | head -n 100 | tail -n1 | awk '{ print length($10) }'
    

In [ ]:
sr_read_lengths

In [ ]:
sr_read_length = sr_read_lengths[0]
sr_read_length

## Long read metrics

In [ ]:
# Sample table exported from Terra. Its coverage and alignment_metrics columns give the Oxford Nanopore
# column of the summary table.
if not os.path.exists('sample-hg38.tsv'):
    !gsutil cp {AOU_LR_DATA}/sample-hg38.tsv sample-hg38.tsv

In [ ]:
f = pd.read_table('sample-hg38.tsv')
f

In [ ]:
# Number of withdrawn participants in the sample table.
f['entity:sample-hg38_id'].astype(int).isin(withdrawn_participants).sum()

In [ ]:
# ONT coverage and median read length (mean +/- SD across samples). alignment_metrics is a stringified dict.
ont_cov_mean = f['coverage'].mean()
ont_cov_std = f['coverage'].std()

ont_cov = f'{ont_cov_mean:.1f} +/- {ont_cov_std:.1f}'

In [ ]:
ont_rl_mean = f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].mean()
ont_rl_std = f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].std()

ont_read_length = f'{ont_rl_mean:,.0f} +/- {ont_rl_std:,.0f}'
ont_read_length

In [ ]:
# Per-sample QC metrics for the long-read (PacBio) callset, with withdrawn participants removed.
if not os.path.exists('auxiliary_metrics.GRCh38.tsv'):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/long_read/aux/auxiliary_metrics.GRCh38.tsv .

In [ ]:
lr_metrics = pd.read_table('auxiliary_metrics.GRCh38.tsv')

In [ ]:
lr_metrics = lr_metrics[~lr_metrics['research_id'].astype(int).isin(withdrawn_participants)]

In [ ]:
lr_metrics

In [ ]:
# Long-read participants by sex at birth (taken from the short-read metrics table).
sr_metrics[sr_metrics['research_id'].isin(lr_metrics['research_id'])]['sex_at_birth'].value_counts(dropna=False)

In [ ]:
lr_female = sr_metrics[sr_metrics['research_id'].isin(lr_metrics.research_id)].query('sex_at_birth == "F"').count()['research_id']
lr_male = sr_metrics[sr_metrics['research_id'].isin(lr_metrics.research_id)].query('sex_at_birth == "M"').count()['research_id']
lr_unknown = len(lr_metrics) - lr_female - lr_male

(lr_female, lr_male, lr_unknown)

In [ ]:
# PacBio column of the summary table: participant count, coverage, and mean aligned read length.
lr_num_samples = f'{lr_metrics.shape[0]:,}'
lr_num_samples

In [ ]:
lr_cov = f'{lr_metrics.mosdepth_cov.mean():.1f} +/- {lr_metrics.mosdepth_cov.std():.1f}'
lr_cov

In [ ]:
lr_read_length = f'{lr_metrics.aligned_read_length_mean.mean():,.0f} +/- {lr_metrics.aligned_read_length_mean.std():,.0f}'
lr_read_length

In [ ]:
# Assembly statistics: one sample's hifiasm report, then all samples' reports concatenated into one file.
!gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/example_sample/report_map.txt | column -t

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/*/report_map.txt | head

In [ ]:
if not os.path.exists('full_report_map.txt'):
    !gcloud storage cat --billing-project=${GOOGLE_PROJECT} gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/*/report_map.txt > full_report_map.txt
    

In [ ]:
# Contigs per haplotype (mean +/- SD), excluding rows containing 'gt_' (thresholded contig counts).
hap1_num = !cat full_report_map.txt | grep '#_contigs' | grep -v 'gt_' | awk '{ print $3 }'
hap2_num = !cat full_report_map.txt | grep '#_contigs' | grep -v 'gt_' | awk '{ print $4 }'

In [ ]:
hap1_num_mean = f'{np.mean([int(a) for a in hap1_num]):,.0f} +/- {np.std([int(a) for a in hap1_num]):,.0f}'
hap1_num_mean

In [ ]:
hap2_num_mean = f'{np.mean([int(a) for a in hap2_num]):,.0f} +/- {np.std([int(a) for a in hap2_num]):,.0f}'
hap2_num_mean

In [ ]:
# Contig N50 per haplotype (median across samples).
hap1_n50 = !cat full_report_map.txt | grep '^N50' | awk '{ print $3 }'
hap2_n50 = !cat full_report_map.txt | grep '^N50' | awk '{ print $4 }'

In [ ]:
hap1_n50_median = f'{np.median([int(a) for a in hap1_n50]):,.0f}'
hap1_n50_median

In [ ]:
hap2_n50_median = f'{np.median([int(a) for a in hap2_n50]):,.0f}'
hap2_n50_median

## Load controlled tier datasets (short read participants)

In [ ]:
# Dataset Builder queries: all short-read participants (used for the self-reported race counts).
# This query represents dataset "Short reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_74841393_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
            )"""

dataset_74841393_person_df = pd.read_gbq(
    dataset_74841393_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_74841393_person_df.head(5)

# This query represents dataset "Short reads" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_74841393_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                SELECT
                    DISTINCT concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (
                        select
                            cast(cr.id as string) as id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                        WHERE
                            concept_id IN (
                                1586134
                            ) 
                            AND domain_id = 'SURVEY'
                    ) a 
                        ON (
                            c.path like CONCAT('%',
                        a.id,
                        '.%')) 
                    WHERE
                        domain_id = 'SURVEY' 
                        AND type = 'PPI' 
                        AND subtype = 'QUESTION'
                    )
            )  
            AND (
                answer.PERSON_ID IN (
                    SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_whole_genome_variant = 1 
                        ) 
                    )
            )"""

dataset_74841393_survey_df = pd.read_gbq(
    dataset_74841393_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_74841393_survey_df.head(5)

# This query represents dataset "Short reads" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_74841393_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string,
        0,
        STRPOS(observation.value_as_string,
        '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
            ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_74841393_zip_code_socioeconomic_df = pd.read_gbq(
    dataset_74841393_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_74841393_zip_code_socioeconomic_df.head(5)

## Load controlled tier datasets (short read participants with EHRs)

In [ ]:
# Dataset Builder query: short-read participants who also have EHR data.
# This query represents dataset "Short reads, basic info" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_00296524_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 
                ) 
            )"""

dataset_00296524_person_df = pd.read_gbq(
    dataset_00296524_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_00296524_person_df.head(5)

In [ ]:
# Drop withdrawn participants.
dataset_00296524_person_df = dataset_00296524_person_df[~dataset_00296524_person_df['person_id'].astype(int).isin(withdrawn_participants)]

In [ ]:
dataset_00296524_person_df.groupby('race').agg('count')

## Load controlled tier datasets (long read participants with EHRs)

In [ ]:
# Dataset Builder queries: long-read participants with EHR data, with survey answers and ZIP3 socioeconomic data.
# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_95752237_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 
                ) 
            )"""

dataset_95752237_person_df = pd.read_gbq(
    dataset_95752237_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_95752237_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                SELECT
                    DISTINCT concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (
                        select
                            cast(cr.id as string) as id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                        WHERE
                            concept_id IN (
                                1586134
                            ) 
                            AND domain_id = 'SURVEY'
                    ) a 
                        ON (
                            c.path like CONCAT('%',
                        a.id,
                        '.%')) 
                    WHERE
                        domain_id = 'SURVEY' 
                        AND type = 'PPI' 
                        AND subtype = 'QUESTION'
                    )
            )  
            AND (
                answer.PERSON_ID IN (
                    SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_whole_genome_variant = 1 
                        ) 
                        AND cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_lr_whole_genome_variant = 1 
                        ) 
                        AND cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_ehr_data = 1 
                        ) 
                    )
                )"""

# dataset_95752237_survey_df = pd.read_gbq(
#     dataset_95752237_survey_sql,
#     dialect="standard",
#     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
#     progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads, basic info, socioeconomic status" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_95752237_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string,
        0,
        STRPOS(observation.value_as_string,
        '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 
                ) 
            ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_95752237_zip_code_socioeconomic_df = pd.read_gbq(
    dataset_95752237_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
# Drop withdrawn participants.
dataset_95752237_person_df = dataset_95752237_person_df[~dataset_95752237_person_df['person_id'].astype(int).isin(withdrawn_participants)]

In [ ]:
dataset_95752237_person_df

In [ ]:
dataset_95752237_zip_code_socioeconomic_df.head(5)

## Load controlled tier datasets (long read participants)

In [ ]:
# Dataset Builder queries: all long-read participants.
# This query represents dataset "Long reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
            )"""

dataset_86956725_person_df = pd.read_gbq(
    dataset_86956725_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                SELECT
                    DISTINCT concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (
                        select
                            cast(cr.id as string) as id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                        WHERE
                            concept_id IN (
                                1586134
                            ) 
                            AND domain_id = 'SURVEY'
                    ) a 
                        ON (
                            c.path like CONCAT('%',
                        a.id,
                        '.%')) 
                    WHERE
                        domain_id = 'SURVEY' 
                        AND type = 'PPI' 
                        AND subtype = 'QUESTION'
                    )
            )  
            AND (
                answer.PERSON_ID IN (
                    SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_whole_genome_variant = 1 
                        ) 
                        AND cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_lr_whole_genome_variant = 1 
                        ) 
                    )
                )"""

# dataset_86956725_survey_df = pd.read_gbq(
#     dataset_86956725_survey_sql,
#     dialect="standard",
#     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
#     progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string,
        0,
        STRPOS(observation.value_as_string,
        '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
            ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_86956725_zip_code_socioeconomic_df = pd.read_gbq(
    dataset_86956725_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
# Drop withdrawn participants.
dataset_86956725_person_df = dataset_86956725_person_df[~dataset_86956725_person_df['person_id'].astype(int).isin(withdrawn_participants)]

In [ ]:
dataset_86956725_person_df.groupby('race').agg(['count'])

## Get sample with EHR counts

In [ ]:
# EHR availability: count and percentage of participants with EHR data, per platform.
sr_ehrs = f'{len(dataset_00296524_person_df):,} ({100.0*len(dataset_00296524_person_df)/sr_metrics.shape[0]:.1f}%)'
sr_ehrs

In [ ]:
lr_ehrs = f'{len(dataset_95752237_person_df):,} ({100.0*len(dataset_95752237_person_df)/len(dataset_86956725_person_df):.1f}%)'
lr_ehrs

In [ ]:
dataset_86956725_person_df

In [ ]:
# ONT participants with EHR data. ont_ids.txt lists the Oxford Nanopore Phase 1 research IDs.
# Oxford Nanopore Phase 1 research IDs.
!gsutil cp {AOU_LR_DATA}/ont_ids.txt ont_ids.txt
# Oxford Nanopore Phase 1 research IDs.
!gsutil cp {AOU_LR_DATA}/ont_ids.txt ont_ids.txt
ont_ids = [line.strip() for line in open("ont_ids.txt") if line.strip()]

num_ont_ehrs = len(list(filter(lambda x: x in set(dataset_95752237_person_df['person_id'].astype('str')), ont_ids)))
ont_ehrs = f'{num_ont_ehrs} ({100.0*num_ont_ehrs/len(ont_ids):.1f}%)'

In [ ]:
# ONT participants by sex at birth.
sr_metrics['research_id'] = sr_metrics['research_id'].astype(str)
ont_metrics = sr_metrics[sr_metrics['research_id'].isin(ont_ids)]

ont_female = ont_metrics.query('sex_at_birth == "F"').count()['research_id']
ont_female = f'{ont_female:,}'

ont_male = ont_metrics.query('sex_at_birth == "M"').count()['research_id']
ont_male = f'{ont_male:,}'

(ont_female, ont_male)

## Get self-reported race stats

In [ ]:
# Self-reported race counts: short-read (q1), PacBio (q2), and ONT (q3) participants.
q1 = dataset_74841393_person_df.groupby('race').agg(['count'])['person_id']
q1

In [ ]:
q2 = dataset_86956725_person_df.groupby('race').agg(['count'])['person_id']
q2

In [ ]:
q2 = dataset_86956725_person_df.groupby('race').agg(['count'])['person_id']
q2

In [ ]:
ont_samples = list(ont_metrics['research_id'])

q3 = dataset_74841393_person_df[dataset_74841393_person_df['person_id'].astype(str).isin(ont_samples)].groupby('race').agg(['count'])['person_id']
q3

In [ ]:
# Combine the race counts into one table with a Total row, one column per platform.
a1 = pd.concat([q1, q2, q3], axis=1, join="inner").fillna(0)
a1.loc['Total'] = a1.sum(numeric_only=True)
a1.columns = ['Illumina', 'PacBio Sequel IIe', 'Oxford Nanopore R10.4']
a1['Description'] = 'self-identified race'
a1['Metric'] = list(a1.index)
a1.iloc[:, :-2] = a1.iloc[:, :-2].applymap(lambda x: f"{int(x):,}" if isinstance(x, (int, float)) else x)
a1.set_index(['Description', 'Metric'], inplace=True)
a1

## Create final table

In [ ]:
# Assemble the dataset summary table (manuscript table): one column per sequencing platform, with the
# race counts appended.
# Define the data as a dictionary'
data = {
    'Description':           ['', 'sex at birth', 'sex at birth', '', '', ''],
    'Metric':                ['participants', 'female', 'male', 'EHRs', 'coverage', 'read length (bp)'],
    'Illumina':              [sr_num_samples, sr_female, sr_male, sr_ehrs, sr_cov, sr_read_length],
    'PacBio Sequel IIe':     [lr_num_samples, lr_female, lr_male, lr_ehrs, lr_cov, lr_read_length],
    'Oxford Nanopore R10.4': ['50', ont_female, ont_male, ont_ehrs, ont_cov, ont_read_length]
}

# Create a DataFrame
df = pd.DataFrame(data)
df.set_index(['Description', 'Metric'], inplace=True)

# This can be directly copy-pasted into tablesgenerator.com for nicer formatting
pd.concat([df, a1])

## Plot metrics

In [ ]:
# Coverage by platform (box plot).
import matplotlib.pyplot as plt

# Creating the box-and-whisker plot
data = [sr_metrics.mean_coverage.tolist(), ont_metrics.mean_coverage.tolist(), lr_metrics.mosdepth_cov.tolist()]
labels = ['Illumina', 'Oxford Nanopore R10.4', 'PacBio Sequel IIe']

#data = [lr_metrics.mosdepth_cov.tolist(), ont_metrics.mean_coverage.tolist(), sr_metrics.mean_coverage.tolist()]
#labels = ['PacBio Sequel IIe', 'Oxford Nanopore R10.4', 'Illumina']


plt.figure(figsize=(7, 3))
plt.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
#plt.title('Coverage Metrics')
plt.ylabel('Coverage', fontsize=14)
#plt.ylabel('Sequencing Platform', fontsize=14)
plt.ylim(0, 50)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig("coverage_metrics_boxplot.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
[int(x) for x in sr_read_lengths]

In [ ]:
lr_metrics.aligned_read_length_mean.tolist()

In [ ]:
f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].tolist()

In [ ]:
# Read length by platform (box plot).
import matplotlib.pyplot as plt

data = [[int(x) for x in sr_read_lengths], f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].tolist(), lr_metrics.aligned_read_length_mean.tolist()]
labels = ['Illumina', 'Oxford Nanopore R10.4', 'PacBio Sequel IIe']

#data = [lr_metrics.aligned_read_length_mean.tolist(), f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].tolist(), [int(x) for x in sr_read_lengths]]
#labels = ['PacBio Sequel IIe', 'Oxford Nanopore R10.4', 'Illumina']

plt.figure(figsize=(7, 3))
plt.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
#plt.title('Coverage Metrics')
plt.xlabel('Mean read length', fontsize=14)
#plt.ylabel('Sequencing Platform', fontsize=14)
#plt.xlim(0, 50)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig("coverage_metrics_boxplot.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
# Combined coverage and read-length panels for the figure (two layouts; the second is the final one).
import matplotlib.pyplot as plt

labels = ['Illumina', 'Oxford Nanopore\nR10.4', 'PacBio\nSequel IIe']

cov_data = [sr_metrics.mean_coverage.tolist(), ont_metrics.mean_coverage.tolist(), lr_metrics.mosdepth_cov.tolist()]
rl_data = [[int(x) for x in sr_read_lengths], f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].tolist(), lr_metrics.aligned_read_length_mean.tolist()]

fig, axs = plt.subplots(2, 1, figsize=(4.5, 5), constrained_layout=True)

axs[0].boxplot(cov_data, labels=['', '', ''], patch_artist=True, showfliers=True)
axs[0].set_ylabel('Coverage', fontsize=14)
axs[0].set_ylim(0, 60)
axs[0].tick_params(axis='both', which='major', labelsize=12)
axs[0].grid(axis='y', linestyle='--', alpha=0.7)

axs[1].boxplot(rl_data, labels=labels, patch_artist=True, showfliers=True)
axs[1].set_ylabel('Mean read length', fontsize=14)
axs[1].set_ylim(0, 25000)
axs[1].tick_params(axis='both', which='major', labelsize=12)
axs[1].grid(axis='y', linestyle='--', alpha=0.7)

fig.align_ylabels(axs)

plt.savefig("metrics_boxplot.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

labels = ['Illumina', 'Oxford Nanopore\nR10.4', 'PacBio\nSequel IIe']

cov_data = [
    sr_metrics.mean_coverage.tolist(),
    ont_metrics.mean_coverage.tolist(),
    lr_metrics.mosdepth_cov.tolist()
]

rl_data = [
    [int(x) for x in sr_read_lengths],
    f['alignment_metrics'].apply(lambda x: ast.literal_eval(x)).apply(pd.Series)['median_read_length'].tolist(),
    lr_metrics.aligned_read_length_mean.tolist()
]

# Slightly reduce vertical space and overall margins
fig, axs = plt.subplots(2, 1, figsize=(5.5, 4.5), gridspec_kw={'hspace': 0.3})

axs[0].boxplot(cov_data, labels=['', '', ''], patch_artist=True, showfliers=True)
axs[0].set_ylabel('Coverage', fontsize=18)
axs[0].set_ylim(0, 60)
axs[0].tick_params(axis='both', which='major', labelsize=14)
axs[0].grid(axis='y', linestyle='--', alpha=0.7)

axs[1].boxplot(rl_data, labels=labels, patch_artist=True, showfliers=True)
axs[1].set_ylabel('Mean read length', fontsize=18)
axs[1].set_ylim(0, 25000)
axs[1].tick_params(axis='both', which='major', labelsize=14)
axs[1].grid(axis='y', linestyle='--', alpha=0.7)

fig.align_ylabels(axs)

# Save with tight bounding box to trim whitespace
plt.savefig("metrics_boxplot.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)

plt.show()